# Fine-tuning Llama 3.1 8B in Google Colab (Memory Optimized)

This notebook imports the entire repository from GitHub and fine-tunes the model with **aggressive memory optimizations** to prevent RAM overflow.

## 🚨 Memory Optimizations Applied

- **Reduced batch size**: 1 (instead of 2)
- **Increased gradient accumulation**: 8 (to maintain effective batch size)
- **Reduced max sequence length**: 256 (instead of 512)
- **Reduced LoRA rank**: 8 (instead of 16)
- **Limited dataset size**: Max 3000 train / 500 val examples
- **Smaller chunk size**: 256 tokens (instead of 512)
- **Aggressive memory clearing**: After each major step

## 📋 Setup Instructions

1. **Replace `YOUR_USERNAME`** in the clone command below with your GitHub username
2. **Enable GPU**: Runtime > Change runtime type > GPU (T4 recommended)
3. **Run cells in order**
4. **Get Hugging Face token**: https://huggingface.co/settings/tokens (optional, only for Llama)
5. **Request Llama access**: https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct (optional)

## 💡 Where to Run This

**Best options for running this:**

1. **Google Colab Pro** ($10/month) - 32GB RAM, better GPUs
2. **Kaggle Notebooks** (Free) - 30GB RAM, P100 GPU
3. **Hugging Face Spaces** (Free tier) - Limited but works
4. **Local machine** with 16GB+ RAM and NVIDIA GPU
5. **AWS/GCP/Azure** - Pay-as-you-go GPU instances

## Step 1: Clone Repository from GitHub

**⚠️ IMPORTANT: Replace `YOUR_USERNAME` with your actual GitHub username!**

If your repo is private, you'll need to use a Personal Access Token.

In [ ]:
# Clone your repository from GitHub
# ⚠️ REPLACE YOUR_USERNAME with your actual GitHub username!
!git clone https://github.com/YOUR_USERNAME/Mystyle-AI.git

# Navigate to the project directory
%cd Mystyle-AI

# Verify files are there
!ls -la

# Show current directory
%pwd

print("✅ Repository cloned successfully!")

## Step 2: Install Dependencies

**Important:** Make sure GPU is enabled: Runtime > Change runtime type > GPU

In [ ]:
# Install all required packages
!pip install transformers datasets peft accelerate bitsandbytes -q
!pip install unsloth -q
!pip install huggingface_hub -q
!pip install requests -q

print("✅ All dependencies installed!")

## Step 3: Setup Hugging Face Token (OPTIONAL)

**⚠️ IMPORTANT:** We're using **Mistral 7B** by default (no approval needed!)

**Only needed if you want to use Llama models:**
1. Create account: https://huggingface.co/ (free, 2 min)
2. Request access: https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct (may be rejected)
3. Get token: https://huggingface.co/settings/tokens (2 min)

**Skip this step** - Mistral 7B works without any token!

In [ ]:
# OPTIONAL: Only needed for Llama models (which require approval)
# Mistral 7B works without this!

# Uncomment below if you want to use Llama models:
# from huggingface_hub import login
# import getpass
#
# hf_token = getpass.getpass("Enter your Hugging Face token (hf_...): ")
# login(token=hf_token)
#
# print("✅ Logged in to Hugging Face!")

print("✅ Skipping Hugging Face login - Using Mistral 7B (no approval needed!)")

## Step 4: Upload Your Data

Upload your writing sample file (`my-writing.txt`).

In [ ]:
from google.colab import files
import os
from pathlib import Path

# Create data directory
os.makedirs('data', exist_ok=True)

# Upload your writing sample
print("Please upload your writing file (my-writing.txt)")
uploaded = files.upload()

# Move to data directory if needed
for filename in uploaded.keys():
    if 'writing' in filename.lower() or filename.endswith('.txt'):
        if filename != 'data/my-writing.txt':
            os.rename(filename, 'data/my-writing.txt')
        print(f"✅ Saved to data/my-writing.txt")
        break
    else:
        # Move any uploaded file to data directory
        os.rename(filename, f'data/{filename}')
        print(f"✅ Saved to data/{filename}")

## Step 5: Process Data (Memory Optimized)

Load and process your writing sample into training datasets with **memory optimizations**.

**Key optimizations:**
- Smaller chunk size (256 instead of 512)
- Limited dataset size (max 3000 train / 500 val examples)
- Aggressive memory clearing

In [ ]:
from src.data_processor import DataProcessor
import gc

# Initialize processor with smaller chunk size to reduce memory
processor = DataProcessor(data_file="data/my-writing.txt", chunk_size=256)  # Reduced from 512

# Process data
train_dataset, val_dataset, stats = processor.process(save_datasets=True)

# MEMORY OPTIMIZATION: Limit dataset size to prevent OOM
# Colab free tier has ~12-15GB RAM, so we limit to ~3000 examples max
MAX_TRAIN_EXAMPLES = 3000  # Reduced from unlimited
MAX_VAL_EXAMPLES = 500     # Reduced from unlimited

if len(train_dataset) > MAX_TRAIN_EXAMPLES:
    print(f"⚠️ Limiting train dataset from {len(train_dataset)} to {MAX_TRAIN_EXAMPLES} examples to save memory")
    train_dataset = train_dataset.select(range(MAX_TRAIN_EXAMPLES))

if len(val_dataset) > MAX_VAL_EXAMPLES:
    print(f"⚠️ Limiting val dataset from {len(val_dataset)} to {MAX_VAL_EXAMPLES} examples to save memory")
    val_dataset = val_dataset.select(range(MAX_VAL_EXAMPLES))

# Clear memory
gc.collect()

# Display statistics
print("\n📊 Data Statistics:")
print(f"Total words: {stats['total_words']:,}")
print(f"Total characters: {stats['total_characters']:,}")
print(f"Train examples: {len(train_dataset)}")
print(f"Val examples: {len(val_dataset)}")
print("\n✅ Data processing complete!")

## Step 6: Load Model (Memory Optimized)

The model will download automatically on first run (~15GB, 15-20 minutes).

**Memory optimizations:**
- LoRA rank reduced to 8 (instead of 16)
- Memory cleared after loading

In [ ]:
from src.finetune import FineTuner
import torch
import gc

# Clear any existing memory
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Initialize fine-tuner (uses Mistral 7B by default - NO APPROVAL NEEDED!)
# Model: unsloth/Mistral-7B-Instruct-v0.3-bnb-4bit
fine_tuner = FineTuner()

# To use Llama instead (requires approval):
# fine_tuner = FineTuner(model_name="unsloth/llama-3.1-8b-bnb-4bit")

# Load model (downloads automatically on first run, ~15GB, takes 15-20 min)
print("🔄 Loading Mistral 7B model (no approval needed!)...")
print("⏳ First time: Downloading model (~15GB, 15-20 minutes)...")
print("✅ Subsequent runs: Uses cache (instant)")
fine_tuner.load_model()

# Setup LoRA with smaller rank to save memory
print("\n🔄 Setting up LoRA (memory optimized)...")
fine_tuner.setup_lora(r=8)  # Reduced from 16 to save memory

# Clear memory after model loading
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("\n✅ Model ready for training!")

## Step 7: Train the Model (Memory Optimized)

Fine-tune the model on your writing style with **aggressive memory optimizations**.

**⚠️ Memory Optimization Settings:**
- Batch size: **1** (reduced from 2)
- Gradient accumulation: **8** (increased to maintain effective batch size)
- Max sequence length: **256** (reduced from 512)
- LoRA rank: **8** (reduced from 16)
- Dataset size limited to prevent OOM

This may take 30-60 minutes.

In [ ]:
import gc
import torch

# Clear memory before training
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Train the model with memory-optimized settings
print("🚀 Starting training with memory optimizations...")
print("⏳ This may take 30-60 minutes depending on your dataset size")
print("\n📊 Memory Optimization Settings:")
print("  - Batch size: 1 (reduced from 2)")
print("  - Gradient accumulation: 8 (increased to maintain effective batch size)")
print("  - Max sequence length: 256 (reduced from 512)")
print("  - LoRA rank: 8 (reduced from 16)")
print("  - Dataset size limited to prevent OOM")

# Use smaller batch size and more gradient accumulation
fine_tuner.train(
    train_dataset, 
    val_dataset, 
    num_epochs=3, 
    batch_size=1,  # Reduced from 2 to save memory
    learning_rate=2e-4,
    warmup_steps=5,
    logging_steps=5,  # Reduced logging frequency
    save_steps=1000,   # Save less frequently
    max_length=256     # Reduced from 512 to save memory
)

# Clear memory after training
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("\n✅ Training complete!")

## Step 8: Save the Model

Save the fine-tuned model for later use.

In [ ]:
# Save the model
print("💾 Saving model...")
fine_tuner.save_model()
# Note: save_for_ollama() uses more memory, so we skip it or do it separately
# fine_tuner.save_for_ollama()

print("\n✅ Model saved successfully!")
print("📁 Location: checkpoints/lora_adapter/")

## Step 9: Setup API for Generation (Instead of Ollama)

Get a free API key from one of these services:
- **Groq** (Recommended): https://console.groq.com/ - 14,400 free requests/day
- **Hugging Face**: https://huggingface.co/settings/tokens - 1,000 free requests/day
- **Together AI**: https://together.ai/ - $25 free credits

## Step 10: Test Text Generation

Test the API-based generation with your fine-tuned model style.

In [ ]:
from src.api_generate import APIStyleGenerator

# Initialize generator with Mistral API (default, already configured!)
generator = APIStyleGenerator(service="mistral")

# Test rewrite
text = "Artificial intelligence has revolutionized technology."
rewritten = generator.rewrite(text)
print("📝 Original:", text)
print("\n✨ Rewritten:", rewritten)

In [ ]:
# Test topic writing
topic = "The importance of education"
generated = generator.write_about(topic)
print(f"📌 Topic: {topic}")
print(f"\n✨ Generated:\n{generated}")

## Step 11: Download Model (Optional)

Download your fine-tuned model to your local machine.

In [ ]:
# Download the fine-tuned model
from google.colab import files
import shutil

# Create zip file
shutil.make_archive('fine_tuned_model', 'zip', 'checkpoints/lora_adapter')

# Download
files.download('fine_tuned_model.zip')

print("✅ Model downloaded to your computer!")

## Step 12: Save Changes to GitHub (Optional)

If you want to save your trained model or changes back to GitHub:

In [ ]:
# Configure Git (first time only)
# !git config --global user.email "your_email@example.com"
# !git config --global user.name "Your Name"

# Add, commit, and push changes
# %cd /content/Mystyle-AI
# !git add .
# !git commit -m "Trained model from Colab"
# !git push

# print("✅ Changes pushed to GitHub!")

print("⚠️ Uncomment the lines above and configure Git to push changes")